---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;"> 

<h1 align="center"><b>Clasificación XGBoost con Scikit-Learn</b></h1> 

---

XGBoost Classifier (*Extreme Gradient Boosting Classifier*) es un algoritmo de aprendizaje supervisado basado en árboles de decisión secuenciales y técnicas de *Gradient Boosting*, diseñado para resolver problemas de clasificación mediante la construcción iterativa de múltiples árboles que corrigen progresivamente los errores de los árboles anteriores. El modelo optimiza una función de pérdida utilizando métodos de gradiente y regularización matemática, permitiendo obtener alta precisión predictiva, control del sobreajuste y gran eficiencia computacional. XGBoost combina técnicas de *boosting*, regularización, muestreo aleatorio y optimización paralela para producir modelos robustos capaces de capturar relaciones complejas y no lineales entre las variables predictoras y la variable objetivo.

La documentación para su implementación puede ser consultada en 🔗 <a href="https://xgboost.readthedocs.io/en/release_3.2.0/"> XGBClassifier</a>


</div>

---

<p align="center">
 <img src="img/xgb_img.png" width="1100" height="650">
</p>

---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

## **Introducción**

XGBoost (*Extreme Gradient Boosting*) es un algoritmo de aprendizaje supervisado basado en árboles de decisión secuenciales y técnicas de *boosting*.

Es uno de los modelos más utilizados en:

- clasificación,
- detección de spam,
- fraude financiero,
- analítica de datos,
- Machine Learning competitivo.

---

### **Idea Fundamental**

A diferencia de Random Forest, donde los árboles son independientes, XGBoost construye árboles de forma secuencial.

Cada nuevo árbol intenta corregir los errores cometidos por los árboles anteriores.

---

### **Modelo General**

El modelo final se construye sumando múltiples árboles:

$$
F_M(x)
=
\sum_{m=1}^{M}
f_m(x)
$$

donde:

- $f_m(x)$ = árbol de decisión número $m$,
- $M$ = número total de árboles.

---

### **Boosting**

El algoritmo agrega árboles gradualmente:

$$
\hat{y}^{(t)}
=
\hat{y}^{(t-1)}
+
f_t(x)
$$

donde:

- $\hat{y}^{(t-1)}$ = predicción previa,
- $f_t(x)$ = nuevo árbol corrector.

---

### **Función Objetivo**

XGBoost minimiza:

$$
\mathcal{L}
=
\sum_{i=1}^{n}
l(y_i,\hat{y}_i)
+
\sum_{k=1}^{K}
\Omega(f_k)
$$

donde:

- $l(y_i,\hat{y}_i)$ = función de pérdida,
- $\Omega(f_k)$ = regularización del árbol.

---

### **Regularización**

La regularización ayuda a evitar sobreajuste:

$$
\Omega(f)
=
\gamma T
+
\frac{1}{2}\lambda ||w||^2
$$

donde:

- $T$ = número de hojas,
- $w$ = pesos de las hojas,
- $\gamma$ = penalización por complejidad,
- $\lambda$ = regularización L2.

---

### **Clasificación Probabilística**

Para clasificación binaria, XGBoost usa una función sigmoide:

$$
P(y=1|x)
=
\frac{1}{1+e^{-F(x)}}
$$

donde:

$$
F(x)
=
\sum_{m=1}^{M}
f_m(x)
$$

---

### **Idea Intuitiva**

Cada árbol nuevo aprende:

$$
\text{Errores anteriores}
$$

y trata de corregirlos.

---

| Categoría | Aspecto | Descripción |
|---|---|---|
| ✅ Ventajas | Alta precisión predictiva | Suele obtener excelente desempeño en problemas de clasificación complejos. |
|  | Excelente desempeño en datasets complejos | Captura patrones difíciles y relaciones avanzadas entre variables. |
|  | Maneja relaciones no lineales | Aprende fronteras de decisión altamente complejas. |
|  | Incluye regularización | Reduce el sobreajuste mediante penalizaciones matemáticas. |
|  | Maneja variables numéricas y categóricas | Puede trabajar con distintos tipos de variables predictoras. |
|  | Muy usado en Kaggle y producción | Es uno de los algoritmos más utilizados en Machine Learning moderno. |
||||
| ❌ Desventajas | Más complejo de interpretar | Es menos interpretable que modelos lineales o árboles simples. |
|  | Puede sobreajustar si no se regula | Requiere ajuste adecuado de hiperparámetros. |
|  | Mayor costo computacional | El entrenamiento puede ser más pesado computacionalmente. |
|  | Sensible a hiperparámetros | El desempeño depende fuertemente de una buena optimización. |

---

### **Implementación**

`XGBClassifier` no pertenece de forma nativa a la biblioteca Scikit-Learn. En realidad, hace parte de la librería independiente **XGBoost**, desarrollada específicamente para implementar algoritmos de *Gradient Boosting* de manera eficiente y optimizada.

Sin embargo, XGBoost proporciona una interfaz completamente compatible con Scikit-Learn, lo que permite utilizar prácticamente la misma sintaxis y flujo de trabajo de modelos tradicionales de `sklearn`, incluyendo métodos como:

- `fit()`
- `predict()`
- `predict_proba()`
- `cross_val_score()`
- `GridSearchCV`
- `RandomizedSearchCV`
- `Pipeline`

Esto facilita integrar `XGBClassifier` dentro de pipelines, validación cruzada y procesos de optimización de hiperparámetros exactamente igual que otros modelos de Scikit-Learn.

Para utilizarlo, primero es necesario instalar la librería XGBoost desde la terminal:



```bash
pip install xgboost
```

En Python se implementa mediante:

```bash
from xgboost import XGBClassifier
```

</div>

---
---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

<h1 align="center"><b>Modelación con Clasificación XGBoost - Spam</b></h1> 

En este proyecto, se construyo un clasificador para predecir la clasificación de un correo electrónico es Spam o no. Se desarrolla modelos basados con el índice de Gini y entropía como criterio. Se emplea el conjunto de datos de correos electrónicos, el cual se encuentra en la carpeta ```ml-project/data/spam```.

</div>

---

In [ ]:
# ============ #
# 1. LIBRERÍAS #
# ============ #

import numpy as np
import pandas as pd
import os
import plotly.express as px
import plotly.graph_objects as go
import joblib
from pathlib import Path
from sklearn.model_selection import (train_test_split, cross_validate, RandomizedSearchCV)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)

In [ ]:
# =================== #
# 2. LECTURA DE RUTAS #
# =================== #

mainpath= "/Users/duvancatano/Documents/Data_Analytics_UdeA/ml-project/data/spam"
filename= "spam_dataset.csv"
fullpath= os.path.join(mainpath,filename)

In [ ]:
# ================= #
# 3. CARGA DE DATOS #
# ================= #

data = pd.read_csv(fullpath, sep=",") # El separador es "," porque en el archivo .csv los valores están separados por coma
pd.set_option('display.max_columns', None) # Para mostrar todas las columnas del DataFrame sin truncar
data.sample(10)

In [ ]:
# ==================================== #
# 4. CODIFICACIÓN DE VARIABLE OBJETIVO #
# ==================================== #

# 1 = Spam
# 0 = No Spam

data["type"] = np.where(

    data["type"] == "spam",

    1,

    0

)


In [ ]:
# ============================================================================== #
# 5. VARIABLES PREDICTORAS Y OBJETIVO
# ============================================================================== #

X = data.drop(columns="type")
y = data["type"]

In [ ]:
# ==================================== #
# 6. VARIABLES NUMÉRICAS Y CATEGÓRICAS #
# ==================================== #

cat_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist() # Aunque no tenemos categóricas, se deja si en el futuro se incluye alguna

num_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist() # Se combina variables de tipo entero y real

In [ ]:
# =================== #
# 7. PREPROCESAMIENTO #
# =================== #

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols) ], 
                                 remainder="passthrough"
                                 )

In [ ]:
# ========================= #
# 8. PARTICIÓN TRAIN - TEST #
# ========================= #

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=123)

In [ ]:
# =========== #
# 9. PIPELINE #
# =========== #

pipeline_xgb = Pipeline([
    ("preprocessing", preprocessor),
    ("model", XGBClassifier(
            eval_metric="logloss",
            random_state=123
        )
    )
])

In [ ]:
# ================= #
# 10. ENTRENAMIENTO #
# ================= #

pipeline_xgb.fit(X_train, y_train)

In [ ]:
# ================ #
# 11. PREDICCIONES #
# ================ #

y_pred = pipeline_xgb.predict(X_test)

y_prob = pipeline_xgb.predict_proba(X_test)[:,1]

In [ ]:
# ============ #
# 12. MÉTRICAS #
# ============ #

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

In [ ]:
# ============== #
# 13. RESULTADOS #
# ============== #

print("="*22)
print("MÉTRICAS XGBOOST")
print("="*22)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

In [ ]:
# ======================= #
# 14. MATRIZ DE CONFUSIÓN #
# ======================= #

cm = confusion_matrix(
    y_test,
    y_pred
)

cm_df = pd.DataFrame(
    cm,
    index=[
        "Real No Spam",
        "Real Spam"
    ],

    columns=[
        "Pred No Spam",
        "Pred Spam"
    ]
)

fig = px.imshow(
    cm_df,
    text_auto=True,
    color_continuous_scale="Reds",
    title="Matriz de Confusión"
)

fig.update_layout(
    template="plotly_dark",
    title_x=0.5,
    height=600

)

fig.show()

In [ ]:
# ========================= #
# 15. CLASSIFICATION REPORT #
# ========================= #

print("")
print("="*22)
print("CLASSIFICATION REPORT")
print("="*22)
print(
    classification_report(
        y_test,
        y_pred
    )
)

In [ ]:
# ============= #
# 16. CURVA ROC #
# ============= #

fpr, tpr, thresholds = roc_curve(
    y_test,
    y_prob
)

fig = px.area(
    x=fpr,
    y=tpr,
    title=f"Curva ROC (AUC = {roc_auc:.4f})"
)

fig.add_shape(
    type="line",
    line=dict(
        dash="dash"
    ),
    x0=0,
    x1=1,
    y0=0,
    y1=1
)

fig.update_layout(
    template="plotly_dark",
    title={
        "text": f"Curva ROC (AUC = {roc_auc:.4f})",
        "x": 0.5,
        "xanchor": "center"
    },
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    height=600
)

fig.show()

---
## **Validación Cruzada**
---

In [ ]:
# ====================== #
# 17. VALIDACIÓN CRUZADA #
# ====================== #

cv_results = cross_validate(
    pipeline_xgb,
    X_train,
    y_train,
    cv=5,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ],
    return_train_score=False
)

In [ ]:
# ================================= #
# 18. RESULTADOS VALIDACIÓN CRUZADA #
# ================================= #

print("")
print("="*18)
print("VALIDACIÓN CRUZADA")
print("="*18)
print(f"Accuracy CV : {cv_results['test_accuracy'].mean():.4f}")
print(f"Precision CV: {cv_results['test_precision'].mean():.4f}")
print(f"Recall CV   : {cv_results['test_recall'].mean():.4f}")
print(f"F1-Score CV : {cv_results['test_f1'].mean():.4f}")
print(f"ROC-AUC CV  : {cv_results['test_roc_auc'].mean():.4f}")

In [ ]:

# ============================================================================== #
# RANDOM SEARCH XGBOOST
# ============================================================================== #

param_dist = {

    "model__n_estimators": [
        100,
        200,
        300,
        500
    ],

    "model__max_depth": [
        3,
        5,
        7,
        10
    ],

    "model__learning_rate": [
        0.01,
        0.05,
        0.1,
        0.2
    ],

    "model__subsample": [
        0.7,
        0.8,
        1.0
    ],

    "model__colsample_bytree": [
        0.7,
        0.8,
        1.0
    ],

    "model__gamma": [
        0,
        0.1,
        0.3,
        1
    ],

    "model__reg_lambda": [
        0,
        1,
        5,
        10
    ]

}


In [ ]:
# ======================== #
# 20. RANDOMIZED SEARCH CV #
# ======================== #

random_search = RandomizedSearchCV(
    estimator=pipeline_xgb,
    param_distributions=param_dist,
    n_iter=30,
    scoring="roc_auc",
    cv=5,
    verbose=1,
    random_state=123,
    n_jobs=-1
)

In [ ]:
# =============================== #
# 21. ENTRENAMIENTO RANDOM SEARCH #
# =============================== #

random_search.fit(
    X_train,
    y_train
)

In [ ]:
# ================ #
# 22. MEJOR MODELO #
# ================ #

best_model = random_search.best_estimator_

In [ ]:
# =========================== #
# 23. MEJORES HIPERPARÁMETROS #
# =========================== #

print("")
print("="*70)
print("MEJORES HIPERPARÁMETROS")
print("="*70)

for key, value in random_search.best_params_.items():

    print(f"{key}: {value}")

In [ ]:
# ============================= #
# 24. PREDICCIONES MEJOR MODELO #
# ============================= #

y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:,1]

In [ ]:
# ========================= #
# 25. MÉTRICAS MEJOR MODELO #
# ========================= #

print("")
print("="*23)
print("MÉTRICAS MEJOR MODELO")
print("="*23)
print(f"Accuracy : {accuracy_score(y_test, y_pred_best):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_best):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_best):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred_best):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_best):.4f}")


In [ ]:
# ============================ #
# 26. IMPORTANCIA DE VARIABLES #
# ============================ #

feature_names = (
    best_model
    .named_steps["preprocessing"]
    .get_feature_names_out()
)

importances = (
    best_model
    .named_steps["model"]
    .feature_importances_
)

importance_df = pd.DataFrame({
    "Variable": feature_names,
    "Importancia": importances
})

importance_df = (
    importance_df
    .sort_values(
        by="Importancia",
        ascending=False
    )
)

In [ ]:
# ==================================== #
# 27. TOP 20 VARIABLES MÁS IMPORTANTES #
# ==================================== #

top_importance = importance_df.head(20)

fig = px.bar(
    top_importance,
    x="Importancia",
    y="Variable",
    orientation="h",
    text="Importancia",
    color="Importancia",
    color_continuous_scale="Reds",
    title="Top 20 Variables Más Importantes"
)

fig.update_layout(
    template="plotly_dark",
    title_x=0.5,
    height=700,
    coloraxis_showscale=False
)

fig.update_yaxes(
    categoryorder="total ascending"
)

fig.update_traces(
    texttemplate="%{text:.4f}",
    textposition="outside"
)

fig.show()

In [ ]:
# =========================================== #
# 28. ENTRENAMIENTO FINAL CON TODOS LOS DATOS #
# =========================================== #

final_model = Pipeline([

    ("preprocessing", preprocessor),

    (

        "model",

        XGBClassifier(

            eval_metric="logloss",

            random_state=123,

            n_estimators=200,

            max_depth=5,

            learning_rate=0.05,

            subsample=0.8,

            colsample_bytree=0.8,

            gamma=0,

            reg_lambda=0,

            n_jobs=-1

        )

    )

])

In [ ]:
# ======================= #
# 29. ENTRENAMIENTO FINAL #
# ======================= #

final_model.fit(X, y)

---

## **Serialización del Mejor Modelo**

---

In [ ]:
# ======================================== #
# 30. CARPETA DONDE SE GUARDARÁ EL .joblib #
# ======================================== #
ruta = Path("/Users/duvancatano/Documents/Data_Analytics_UdeA/ml-project/models")

# Crear carpeta sino existe
ruta.mkdir(parents=True, exist_ok=True)

In [ ]:
# ======================== #
# 31. GUARDAR MODELO FINAL #
# ======================== #

joblib.dump(final_model, ruta / "model_xgboost_classifier_spam.joblib")


In [ ]:
# ===================== #
# 32. GUARDAR VARIABLES #
# ===================== #

joblib.dump(X.columns.tolist(), ruta / "features_xgboost_classifier_spam.joblib")

---

# 🎬 **¡FIN!**

---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">    </div>